# Pitch keypoint FINE-TUNE (SNGS domain) — A100

Two stages: **smoke test** (~5 min, verifies plumbing) then **main run** (~2 h).
Upload `dataset.zip` to Drive root *before* connecting to the runtime.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive'

In [ ]:
!pip install -q ultralytics

In [ ]:
!mkdir -p /content/pitch
!unzip -q -o "{DRIVE_DIR}/pitch_ft_dataset.zip" -d /content/pitch
!ls /content/pitch/dataset

In [ ]:
import glob

data_yaml = '/content/pitch/dataset/data.yaml'
print(open(data_yaml).read())
for split in ['train', 'valid']:
    imgs = glob.glob(f'/content/pitch/dataset/{split}/images/*.jpg')
    lbls = glob.glob(f'/content/pitch/dataset/{split}/labels/*.txt')
    print(split, len(imgs), 'images,', len(lbls), 'labels')
assert imgs and lbls, 'dataset missing — check unzip cell'

## Stage 1 — Smoke test (~5 min)
If this passes and loss drops, start the main run. Check epoch time in the log:
projected main-run total ≈ epoch_time × 3.5 (bigger model, more epochs).

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/pitch_best.pt')
model.train(
    data=data_yaml,
    epochs=3,
    imgsz=640,
    batch=-1,
    fraction=0.05,
    cache='disk',
    workers=8,
    project='/content/runs',
    name='pitch_ft_smoke',
)

## Stage 2 — Main run (~2 h)
Checkpoints write to local disk; only `best.pt` is copied to Drive at the end.

In [ ]:
model = YOLO('/content/drive/MyDrive/pitch_best.pt')
model.train(
    data=data_yaml,
    epochs=25,
    imgsz=960,
    batch=-1,
    patience=10,
    lr0=0.002,
    cache='disk',
    workers=8,
    project='/content/runs',
    name='pitch_ft',
)

In [ ]:
import shutil

metrics = model.val()
print('box mAP50:', metrics.box.map50)
print('pose mAP50:', metrics.pose.map50)
print('pose mAP50-95:', metrics.pose.map)
shutil.copy('/content/runs/pitch_ft/weights/best.pt', f'{DRIVE_DIR}/pitch_best_ft.pt')
print('saved to', f'{DRIVE_DIR}/pitch_best_ft.pt')